# Laboratorio 05.1 — Representaciones vectoriales y embeddings

Transformaremos documentos en vectores TF-IDF y construiremos una representación densa de baja dimensión con SVD para observar cómo el corpus afecta la similitud.

## 1. Objetivos de aprendizaje

Al finalizar podrás:
- Interpretar un documento como un vector y calcular similitud coseno.
- Diferenciar una representación léxica TF-IDF de un embedding denso latente.
- Proyectar vectores con SVD y recuperar los vecinos más cercanos.
- Analizar límites de vocabulario, tamaño del corpus y dimensión.

## 2. Conceptos

Un embedding representa una entrada como vector para permitir operaciones geométricas. TF-IDF produce vectores dispersos basados en términos; por sí solo no entiende sinónimos. SVD reduce la matriz término-documento a un espacio denso de factores latentes. La similitud coseno compara orientación: $\cos(x,y)=\frac{x\cdot y}{\|x\|\|y\|}$. Este embedding SVD se aprende del corpus pequeño del laboratorio, no es un modelo neuronal ni un sustituto de embeddings semánticos preentrenados.

## 3. Arquitectura

```mermaid
flowchart LR
    A[Documentos locales] --> B[TF-IDF disperso]
    B --> C[Matriz termino-documento]
    C --> D[SVD de baja dimension]
    E[Consulta] --> F[Proyeccion al mismo espacio]
    D --> G[Similitud coseno]
    F --> G
    G --> H[Vecinos mas cercanos]
```

## 4. Preparación del entorno

Utilizamos scikit-learn para construir matrices y SVD, numpy/pandas para inspección y funciones de similitud reutilizables. Todo se calcula localmente.

In [1]:
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
import sys

raices_candidatas = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
raiz_curso = next(
    (
        candidato
        for raiz in raices_candidatas
        for candidato in (raiz / 'genai', raiz)
        if (candidato / 'src' / 'genai').is_dir()
    ),
    None,
)
if raiz_curso is None:
    raise FileNotFoundError('No se encontró el proyecto genAI con src/genai.')

sys.path.insert(0, str(raiz_curso / 'src'))

In [3]:
import re

import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

from genai.environment import preparar_entorno
from genai.evaluation.metrics import hit_rate_at_k
from genai.retrieval.vector_search import cosine_similarity, rank_by_cosine

raiz_curso = preparar_entorno()

## 5. Definición del problema

Representaremos documentos sintéticos de soporte sobre envíos, devoluciones, pagos y cuentas. El conjunto compartido vive en `data/documents/faq.jsonl`; las etiquetas temáticas se declaran como referencia, no son etiquetas aprendidas por el vectorizador.

## 6. Línea base

La línea base cuenta términos compartidos entre consulta y documento. Es fácil de interpretar pero depende de coincidencias literales; dos frases equivalentes con vocabulario diferente pueden recibir poca puntuación.

In [4]:
documentos = pd.read_json(raiz_curso / 'data' / 'documents' / 'faq.jsonl', lines=True).to_dict(orient='records')

def similitud_lexica(consulta: str, texto: str) -> int:
    """Cuenta terminos distintos compartidos como linea base."""
    terminos_consulta = set(re.findall(r'\w+', consulta.casefold(), flags=re.UNICODE))
    terminos_texto = set(re.findall(r'\w+', texto.casefold(), flags=re.UNICODE))
    return len(terminos_consulta & terminos_texto)

consulta = '¿Cómo puedo rastrear mi paquete?'
tabla_base = pd.DataFrame([
    {'id': documento['id'], 'coincidencias': similitud_lexica(consulta, documento['texto'])}
    for documento in documentos
])
display(tabla_base.sort_values('coincidencias', ascending=False))

,id,coincidencias
0,envio_seguimiento,1
1,envio_entrega,0
2,devolucion_plazo,0
3,devolucion_reembolso,0
4,pago_duplicado,0
5,pago_pendiente,0
6,cuenta_contrasena,0
7,cuenta_datos,0


## 7. Implementación

TF-IDF asigna coordenadas a términos y normaliza cada documento. Mostramos el vocabulario y la matriz para hacer visible qué significa convertir texto en un vector; términos ausentes del vocabulario no aportan coordenadas.

In [5]:
textos = [documento['texto'] for documento in documentos]
vectorizador = TfidfVectorizer(
    lowercase=True,
    stop_words=['el', 'la', 'los', 'las', 'de', 'del', 'un', 'una', 'en', 'y', 'si'],
)
matriz_tfidf = vectorizador.fit_transform(textos)
print('Forma de la matriz TF-IDF:', matriz_tfidf.shape)
print('Primeros terminos:', vectorizador.get_feature_names_out()[:12])
display(pd.DataFrame(matriz_tfidf.toarray(), columns=vectorizador.get_feature_names_out(), index=[d['id'] for d in documentos]).iloc[:, :12])

Forma de la matriz TF-IDF: (8, 105)
Primeros terminos: ['30' 'abrirse' 'accesorios' 'actualizar' 'al' 'algunos' 'ambos' 'aparece'
 'aparecer' 'autoriza' 'banco' 'caduca']


,30,abrirse,accesorios,actualizar,al,algunos,ambos,aparece,aparecer,autoriza,banco,caduca
envio_seguimiento,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
envio_entrega,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.263161,0.000000,0.000000,0.000000,0.000000
devolucion_plazo,0.271584,0.000000,0.271584,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
devolucion_reembolso,0.000000,0.000000,0.000000,0.000000,0.252078,0.211261,0.000000,0.000000,0.000000,0.000000,0.211261,0.000000
pago_duplicado,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.259854,0.000000,0.000000,0.000000,0.000000,0.000000
pago_pendiente,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.246094,0.246094,0.206246,0.000000
cuenta_contrasena,0.000000,0.226386,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.226386
cuenta_datos,0.000000,0.000000,0.000000,0.308432,0.000000,0.258490,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


A continuación proyectamos los vectores dispersos a tres dimensiones latentes. Cambiar la dimensión modifica la compresión y puede perder información; fijamos la semilla para repetir el ajuste en este entorno.

In [6]:
dimension_latente = 3
proyector = TruncatedSVD(n_components=dimension_latente, random_state=17)
embeddings_documentos = proyector.fit_transform(matriz_tfidf)
tabla_embeddings = pd.DataFrame(
    embeddings_documentos,
    index=[documento['id'] for documento in documentos],
    columns=[f'dimension_{indice}' for indice in range(dimension_latente)],
)
print(f'Varianza explicada acumulada: {proyector.explained_variance_ratio_.sum():.3f}')
display(tabla_embeddings)

Varianza explicada acumulada: 0.339


,dimension_0,dimension_1,dimension_2
envio_seguimiento,0.389101,0.126251,-0.167530
envio_entrega,0.499935,-0.324690,-0.382446
devolucion_plazo,0.603305,-0.325382,0.307567
devolucion_reembolso,0.420252,0.507500,0.190265
pago_duplicado,0.232681,0.320875,-0.174277
pago_pendiente,0.262164,0.647846,0.170844
cuenta_contrasena,0.247884,-0.316238,0.708829
cuenta_datos,0.434818,-0.160727,-0.434841


## 8. Experimentos

Comparamos TF-IDF y SVD para dos consultas de envío. La consulta cambia entre casos, pero los documentos, vectorizador y dimensión permanecen fijos. Los resultados muestran similitud, no probabilidad de relevancia.

In [7]:
consultas = [
    {'texto': '¿Cómo puedo rastrear mi paquete?', 'relevantes': {'envio_seguimiento', 'envio_entrega'}},
    {'texto': '¿Dónde consulto el seguimiento de la entrega?', 'relevantes': {'envio_seguimiento', 'envio_entrega'}},
]
filas_busqueda = []
for caso, datos in enumerate(consultas, start=1):
    vector_consulta = vectorizador.transform([datos['texto']]).toarray()[0]
    embedding_consulta = proyector.transform(vectorizador.transform([datos['texto']]))[0]
    ranking_svd = rank_by_cosine(embedding_consulta, embeddings_documentos, top_k=3)
    ranking_tfidf = rank_by_cosine(vector_consulta, matriz_tfidf.toarray(), top_k=3)
    ids_por_indice = [documento['id'] for documento in documentos]
    for metodo, ranking in (('tfidf', ranking_tfidf), ('svd', ranking_svd)):
        ids = [ids_por_indice[indice] for indice, _ in ranking]
        filas_busqueda.append({
            'consulta_id': caso,
            'metodo': metodo,
            'top_1': ids[0],
            'hit_rate_at_3': hit_rate_at_k(ids, datos['relevantes'], 3),
            'similitud_top_1': ranking[0][1],
        })
resultados_embeddings = pd.DataFrame(filas_busqueda)
display(resultados_embeddings)
resultados_embeddings.to_csv(raiz_curso / 'outputs' / '05_01_embedding_search.csv', index=False)

,consulta_id,metodo,top_1,hit_rate_at_3,similitud_top_1
0,1,tfidf,envio_seguimiento,1.0,0.256891
1,1,svd,envio_seguimiento,1.0,0.996195
2,2,tfidf,envio_entrega,1.0,0.283330
3,2,svd,envio_entrega,1.0,0.963754


## 9. Evaluación

Medimos Hit Rate@3 frente a etiquetas relevantes declaradas. Inspeccionamos además la varianza explicada para entender cuánta información retiene la proyección. Un corpus pequeño puede producir embeddings latentes inestables y no capturar sinonimia robusta.

In [8]:
resumen_embedding = resultados_embeddings.groupby('metodo', as_index=False).agg(
    hit_rate_medio=('hit_rate_at_3', 'mean'),
    similitud_media_top_1=('similitud_top_1', 'mean'),
)
display(resumen_embedding)
resumen_embedding.to_csv(raiz_curso / 'outputs' / '05_01_embedding_metrics.csv', index=False)

,metodo,hit_rate_medio,similitud_media_top_1
0,svd,1.0,0.979975
1,tfidf,1.0,0.270111


## 10. Discusión

Los embeddings convierten texto en geometría utilizable para búsqueda, pero las coordenadas dependen del método y los datos. TF-IDF es interpretable y léxico; SVD crea factores densos a partir de coocurrencia en este corpus. Los modelos neuronales preentrenados requieren datos, recursos y evaluación adicionales; no deben asumirse instalados ni equivalentes a este ejemplo.

## 11. Ejercicios

1. Añade documentos de una nueva categoría y observa cómo cambia el vocabulario.
2. Compara dimensiones latentes 2, 3 y 4, manteniendo fija la semilla.
3. Escribe una consulta con sinónimos que TF-IDF no comparta literalmente.
4. Prueba un texto con vocabulario fuera del corpus y registra el vector resultante.

## 12. Desafío

Amplía el corpus y compara TF-IDF, SVD y un embedding de frases preentrenado opcional. Mide recuperación en consultas etiquetadas y registra claramente el modelo de embedding y sus requisitos de descarga.

## 13. Ideas clave

- Los vectores permiten medir similitud y ordenar vecinos.
- TF-IDF representa coincidencia léxica; SVD induce factores latentes locales.
- Dimensión, vocabulario y corpus afectan la geometría.
- Un embedding útil debe evaluarse sobre tareas y consultas representativas.